**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Sigma-Delta & Quantization Theory

How a **1-bit** converter delivers 16+ bits of audio: oversampling spreads quantization noise thin, and the ΣΔ loop *shapes* it out of the band you care about. The bridge between [Real-Time DSP's](./Real_Time_DSP.ipynb) fixed-point world and actual converter hardware — with the 6 dB/bit and noise-shaping laws measured, not recited.

## 1. Pre-requisites

[Real-Time DSP](./Real_Time_DSP.ipynb) S1 (Q-format), [FoSP2](./Foundations_of_Signal_Processing_2.ipynb) S2 (oversampling/decimation), [Statistical SP](./Statistical_Signal_Processing.ipynb) S2 (PSD).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Quantization Noise & the Oversampling Dividend* (~40 min)
**Goal:** the white-noise model of quantization, its 6 dB/bit law, and +3 dB per octave of oversampling.
**Builds on:** [Real-Time DSP](./Real_Time_DSP.ipynb) S1. &nbsp; **Feeds into:** Session 2 (noise shaping).

---

## 2. Noise You Can Dilute

💡 **Intuition.** Quantization error behaves like white noise of total power $\Delta^2/12$ — and that total is **fixed by the step size**, not by the sampling rate. Sample $M×$ faster and the same noise power spreads over $M×$ the bandwidth: the slice sitting on your signal band shrinks by $M$ — **+3 dB of SNR per octave of oversampling** (half a bit per octave), redeemed by a [decimating low-pass filter](./Foundations_of_Signal_Processing_2.ipynb). Useful, but slow: 16 bits from 1 bit would need $2^{30}$× oversampling. Session 2 does radically better.

In [ ]:
# the two laws, measured
    # measure noise power INSIDE the audio band only

# YOUR CODE HERE


**What just happened.** Two laws measured — and both deviate from the printed theory, which is the more instructive outcome.

**The 6 dB/bit law, and its 0.9 dB offset.** Measured 25.5 / 49.1 / 73.1 dB against a printed rule predicting 25.8 / 49.9 / 74.0. Every row falls about 0.9 dB short, and the constant offset is the giveaway: $6.02b + 1.76$ assumes a sine that *fills full scale*, and ours has amplitude 0.9. That costs exactly $20\log_{10}(0.9) = -0.92$ dB. Correct the rule to $6.02b + 0.84$ and the predictions become 24.9 / 49.0 / 73.1 — matching the measurements to 0.6 dB at 4 bits, 0.1 dB at 8, and 0.01 dB at 12.

The shrinking residual is itself informative. The $\Delta^2/12$ white-noise model assumes the error is uniformly distributed across a step and uncorrelated between samples, which becomes truer as steps get finer. At 4 bits there are only 16 levels and the model is visibly approximate; by 12 bits it is exact to a hundredth of a decibel. **The step-per-bit law is solid; the additive constant depends entirely on conventions you must check.** If you also did [Real-Time DSP](./Real_Time_DSP.ipynb), note this is the same lesson with the sign reversed — there the measurement came out 2.93 dB *above* the rule, for the same category of reason.

**The oversampling law, which over-performs.** Measured in-band noise: −53.1 dB at 1×, −60.0 at 4×, −70.6 at 16×. That is a shift of −6.9 dB and −17.5 dB, against −6.0 and −12.0 predicted. The 4× row is close; the 16× row beats its prediction by **5.5 dB**, and the amplitude correction above does not explain it.

Worth being clear that this is not fully accounted for here. The most likely cause is the white-noise model failing under heavy oversampling: at 16× the sine is sampled at 768 kHz, so it moves less than one quantization step between consecutive samples, the error becomes strongly correlated rather than white, and it concentrates into harmonics of 997 Hz instead of spreading uniformly — with some of that energy landing above the audio band and escaping the in-band sum. That is a plausible mechanism, not a verified one. Plotting the error spectrum at 1× and 16× would settle it, and doing so is a better exercise than accepting the explanation.

**What survives regardless.** Oversampling genuinely does dilute in-band quantization noise, and the direction and rough magnitude hold. But +3 dB per octave is a hopeless route to high resolution on its own: reaching 16 bits from 1 bit would need about $2^{30}$× oversampling. The noise budget is fixed and spreading it thinner has sharply diminishing returns. Session 2 stops spreading the noise and starts *moving* it.

---
### 🕐 Session 2 of 2 — *Noise Shaping: the ΣΔ Loop* (~40 min)
**Goal:** put the quantizer in a feedback loop: same noise total, pushed out of band — 1 bit becomes hi-fi.
**Builds on:** Session 1.

---

## 3. The Loop That Cheats

💡 **Intuition.** Wrap the quantizer in feedback: integrate the error before quantizing, subtract the output. Solve the loop and the signal passes untouched while the quantization noise is multiplied by $(1 - z^{-1})$ — a **high-pass**: near DC the loop's memory cancels its own past mistakes. Total noise unchanged; its *location* moved to high frequencies you were going to [decimate away anyway](./Foundations_of_Signal_Processing_2.ipynb). First-order shaping buys 9 dB/octave; second-order, 15 — which is how a 1-bit stream at 64× oversampling delivers CD-quality audio (DSD, and virtually every audio ADC/DAC you own).

In [ ]:

# YOUR CODE HERE


**What just happened.** Three error spectra from the same 1-bit quantizer. The plain quantizer's error is roughly flat — noise spread evenly, including right across the audio band. The first-order ΣΔ curve tilts upward with frequency, sitting well below the flat line at low frequencies and above it at high ones. The second-order curve tilts *harder*: even lower in the audio band, even higher outside it.

**Nothing was removed — read the curves as a redistribution.** Where the shaped traces dip below the flat one at low frequency, they rise above it at high frequency, and the total area is not reduced (second-order shaping slightly *increases* total error power). That is the point: the noise budget from Session 1 is conserved, and the loop only chooses where to spend it. Solving the loop shows why — the output is signal $+\,(1 - z^{-1})e$ for first order, so the signal transfer function is 1 while the noise gets multiplied by a high-pass with a zero at DC. The integrator remembers the error it just made and cancels it next sample, which works near DC and not at all near Nyquist.

**The dotted line is the whole argument.** Everything to its right gets thrown away by the decimator, so noise pushed there costs nothing. Oversampling and noise shaping are therefore not two independent tricks — shaping is only useful *because* there is out-of-band room to dump into, and that room is what 64× oversampling bought. Neither works alone: Session 1 showed oversampling alone needs $2^{30}$×, and shaping without oversampling would have nowhere to put the noise.

Note the steepness ordering — second order tilts more than first, and that slope is the effective-bits number. First-order shaping is worth 9 dB per octave of oversampling, second-order 15, against oversampling's bare 3. That is the difference between hopeless and shipping.

The next cell stops reading slopes off a plot and does the honest thing: run each stream through a real decimator and measure the audio-band SNR that actually results.

In [ ]:
# redeem the promise: decimate each 1-bit stream back to 48 kHz and measure audio-band SNR

# YOUR CODE HERE


**What just happened.** The promise, redeemed and measured after actual decimation back to 48 kHz:

| | audio-band SNR | effective bits |
|---|---|---|
| plain 1-bit @ 64× | **−5.8 dB** | −1.3 |
| ΣΔ 1st order | **46.7 dB** | 7.5 |
| ΣΔ 2nd order | **69.5 dB** | 11.3 |

**Start with the negative number, because it is the strongest result here.** Plain 1-bit at 64× oversampling gives −5.8 dB — *more error power than signal power*, worse than useless. Oversampling by 64 on its own bought essentially nothing, exactly as Session 1's +3 dB/octave arithmetic predicted it would. This row is what makes the experiment controlled: same single bit, same 64× rate, same decimator, and the only difference in the rows below is the feedback loop. That loop is worth **75 dB**.

**One physical bit, eleven effective ones.** The comparator still answers a single yes/no question per sample. The extra resolution is not in the hardware — it is manufactured by feedback and filtering, then collected by the decimator. That is the conceptual payoff of the workshop: a hard *analog* problem (build a precise multi-level converter with matched components) was traded for a fast 1-bit comparator plus a *digital* filter, and digital filters are cheap and exact while analog precision is neither. This is why converter datasheets read like DSP homework, and why the [FPGA workshop's](../Intro_FPGA/Intro_FPGA.ipynb) CIC decimators exist.

**Why not simply keep raising the order?** Stability. Modulators above second order are only conditionally stable and can fall into large-amplitude oscillation from which the loop does not recover. Production designs handle this with careful coefficient scaling or with MASH structures that cascade stable low-order stages rather than building one high-order loop. The 9 → 15 dB/octave progression is real but it does not extrapolate freely.

**And the honest gap.** 11.3 effective bits is not the 16–24 a real audio converter delivers. Closing that gap takes higher-order loops, multi-bit internal quantizers, and **dither** — deliberately added noise that breaks up the correlated-error idle tones flagged in Session 1, which are audible as whistles on quiet passages and are exactly the failure of the white-noise model that made that session's 16× row misbehave. The demo shows the mechanism at work, not the state of the art, and the second-order loop here carries none of the coefficient scaling a shipping design would need.

## 4. Conclusion

Quantization noise has a fixed budget; oversampling dilutes it (+3 dB/octave, measured), and the ΣΔ loop *relocates* it (effective bits measured climbing with loop order). The dirty analog problem became a [multirate filtering](./Foundations_of_Signal_Processing_2.ipynb) problem — which is why converter datasheets read like DSP homework.

---
## Where next

- [Real-Time DSP](./Real_Time_DSP.ipynb) — where the decimated samples land.
- [Intro to FPGA](../Intro_FPGA/Intro_FPGA.ipynb) — CIC decimators: the hardware that does this at GHz.
- [Model Compression](../Intro_Mach_Learn/Model_Compression.ipynb) — the same quantization mathematics, aimed at neural weights.